In [ ]:
from pysat.formula import CNF
from pysat.process import Processor
from pysat.solvers import Solver
from pysat.solvers import Kissat404
from pysat.solvers import Glucose42
from pysat.solvers import Cadical300
from partitionsolver.utils import file_reader
from multiprocessing import Process, Queue
import time
import os
import json

from time import sleep
import matplotlib.pyplot as plt
import matplotlib as mpl

import builtins
import functools
import random

from partitionsolver.partitioning.hypergraph_edges import HyperGraphEdges
from partitionsolver.partitioning.dual_graph import DualGrapSplit

from partitionsolver.utils import timeout_solver

from partitionsolver.solver.partition_solver_dpll import PartitionDPLL
from partitionsolver.solver.partition_solver_cdcl import PartitionCDCL
from partitionsolver.solver.partition_propagator import PartitionPropagator



In [ ]:
builtins.print = functools.partial(print, flush=True)
random.seed(42)

#cnf_file_location = "./instances/instances_small/d6afa5689d75e37111656db8980dd54b-grs-160-48.cnf.xz"
#cnf_file_location = "./instances/instances_small/4c3001f8073986116d98084dde70da05-fsf-300-354-2-2-3-2.9.opt.cnf.xz"
#cnf_file_location = "./instances/instances_small/ad9eb96bac59319fc2f7daffd1f961f8-AProVE07-21.cnf.xz"
#cnf_file_location = "./instances/instances_small/77a0d54f2fb3740a9a321623c0c10f3e-tseitin_grid_n12_m12.cnf.xz"
#cnf_file_location = "./instances/instances_small/b628043a07c5576dd6cd21c9d73a69e0-fixedbandwidth-eq-37_shuffled.cnf.xz"
#cnf_file_location = "./instances/custom/two_random_connect8.cnf"
cnf_file_location = "../instances/random_sat/uf50-028.cnf"

In [ ]:
import signal
from contextlib import contextmanager

@contextmanager
def timeout(seconds):
    def _handler(signum, frame):
        raise TimeoutError(f"timed out after {seconds}s")
    
    old_handler = signal.signal(signal.SIGALRM, _handler)
    signal.alarm(seconds)
    try:
        yield
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old_handler)

In [ ]:

def solve_instance(file_location: str):
    # === Read the clauses from file ===
    num_vars, num_clauses, clauses = file_reader.read_cnf(file_location)

    # === Proprocessor ===
    clauses_all = [clause.tolist() for clause in clauses]
    processor = Processor(bootstrap_with=clauses_all)
    processed_cnf = processor.process()
    if processor.status == False:
        return
    processed_num_vars = processed_cnf.nv
    processed_num_clauses = len(processed_cnf.clauses)
    
    # === Partition Solver ===
    try:
        with timeout(60):
            split = HyperGraphEdges(file_location=file_location, splits_amount=2)
            #split = DualGrapSplit(file_location=file_location, splits_amount=4)
            #split = DualGrapSplit(file_location=file_location, splits_amount=2, cluster_type='community')
            #formulas, comm_variables = split.split_formula(clauses_all, num_vars, num_clauses)
            formulas, comm_variables = split.split_formula(processed_cnf.clauses, processed_num_vars, processed_num_clauses)
    except TimeoutError:
        print(f"skipping {file_location}: took too long")
        return


    try:
        with timeout(60):
            prop_solver = PartitionPropagator(processed_num_vars, comm_variables, formulas, debug_level=0)
            sat = prop_solver.solve()
    except TimeoutError:
        print(f"skipping {file_location}: took too long")
        sat = "unknown"

    print(f"Satisfiable: {sat}")




#directory = "../instances/test/unsat_problem"
#directory = "../instances/test/unsat_random"

max_entries = 200
for entry in os.scandir("../instances/test/benchmarks"):
    max_entries -= 1
    if max_entries < 0:
        break
    print(f"== Next: {entry.name}")
    solve_instance(entry.path)
